# TerraMind Holdout Review

This notebook is designed for presentation and review.

It loads the latest successful **large-run** holdout evaluation from `data/experiments/dev-runs`, summarizes the run, and visualizes:
- Sentinel-2 RGB before image
- Sentinel-2 RGB after image
- ground-truth damage labels
- TerraMind prediction
- agreement/error map

The goal is to make qualitative review easy for PI and EO-team discussion.

In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio as rio
from IPython.display import Markdown, display

PROJECT_ROOT = Path('/users/PGS0218/julina/projects/geography/damage_mapping_terramind/V2')
LARGE_RUNS_DIR = PROJECT_ROOT / 'data/experiments/dev-runs'
RUN_NAME = None  # Example: 'terramind_2026-03-11_192131'


def find_latest_large_run(runs_dir: Path) -> Path:
    candidates = []
    for run_dir in sorted(runs_dir.glob('terramind_*')):
        if (run_dir / 'metrics.txt').exists() and (run_dir / 'geotiffs').exists():
            candidates.append(run_dir)
    if not candidates:
        raise FileNotFoundError(f'No successful large runs found in {runs_dir}')
    return candidates[-1]


run_dir = LARGE_RUNS_DIR / RUN_NAME if RUN_NAME else find_latest_large_run(LARGE_RUNS_DIR)
pred_dir = run_dir / 'geotiffs'
holdout_before_dir = PROJECT_ROOT / 'data/input/Images_large/Test/Before/S2L2A'
holdout_after_dir = PROJECT_ROOT / 'data/input/Images_large/Test/After/S2L2A'
holdout_label_dir = PROJECT_ROOT / 'data/input/Images_large/Test/Labels'

pred_files = sorted(pred_dir.glob('predicted_map_*_colored.tif'))
label_files = sorted(holdout_label_dir.glob('*.tif'))
before_files = sorted(holdout_before_dir.glob('*.tif'))
after_files = sorted(holdout_after_dir.glob('*.tif'))

assert len(pred_files) == len(label_files) == len(before_files) == len(after_files), 'Holdout file counts do not match.'

display(Markdown(f'**Latest large-run directory:** `{run_dir}`'))
display(Markdown(f'**Holdout tiles available:** `{len(pred_files)}`'))

In [ ]:
CLASS_NAMES = {
    0: 'Background',
    1: 'No damage',
    2: 'Damage',
    3: 'Possible damage',
}

CLASS_COLORS = {
    0: np.array([0, 0, 0], dtype=np.uint8),
    1: np.array([34, 139, 34], dtype=np.uint8),
    2: np.array([220, 20, 60], dtype=np.uint8),
    3: np.array([255, 191, 0], dtype=np.uint8),
}

ERROR_COLORS = {
    0: np.array([0, 0, 0], dtype=np.uint8),
    1: np.array([245, 245, 245], dtype=np.uint8),
    2: np.array([0, 122, 255], dtype=np.uint8),
}


def parse_metrics(metrics_path: Path) -> pd.DataFrame:
    text = metrics_path.read_text()
    pattern = re.compile(
        r'Image\s+(?P<image_id>\d+)\s+metrics:\s+'
        r'Accuracy:\s+(?P<Accuracy>[0-9.]+)\s+'
        r'Precision:\s+(?P<Precision>[0-9.]+)\s+'
        r'Recall:\s+(?P<Recall>[0-9.]+)\s+'
        r'F1:\s+(?P<F1>[0-9.]+)\s+'
        r'IoU:\s+(?P<IoU>[0-9.]+)',
        re.MULTILINE,
    )
    records = []
    for match in pattern.finditer(text):
        row = match.groupdict()
        row['image_id'] = int(row['image_id'])
        for key in ('Accuracy', 'Precision', 'Recall', 'F1', 'IoU'):
            row[key] = float(row[key])
        records.append(row)
    return pd.DataFrame(records).sort_values('image_id').reset_index(drop=True)


def read_single_band(path: Path) -> np.ndarray:
    with rio.open(path) as src:
        return src.read(1)


def colorize_labels(label: np.ndarray, color_table: dict[int, np.ndarray]) -> np.ndarray:
    rgb = np.zeros(label.shape + (3,), dtype=np.uint8)
    for cls, color in color_table.items():
        rgb[label == cls] = color
    return rgb


def percentile_stretch(rgb: np.ndarray, lower: float = 2.0, upper: float = 98.0) -> np.ndarray:
    rgb = rgb.astype(np.float32)
    out = np.zeros_like(rgb, dtype=np.float32)
    for channel in range(rgb.shape[-1]):
        band = rgb[..., channel]
        lo, hi = np.percentile(band, [lower, upper])
        if hi <= lo:
            out[..., channel] = 0.0
        else:
            out[..., channel] = np.clip((band - lo) / (hi - lo), 0.0, 1.0)
    return out


def read_s2_rgb(path: Path) -> np.ndarray:
    with rio.open(path) as src:
        descriptions = list(src.descriptions)
        band_lookup = {name: idx + 1 for idx, name in enumerate(descriptions) if name}
        required = ['B4', 'B3', 'B2']
        missing = [name for name in required if name not in band_lookup]
        if missing:
            raise ValueError(f'{path.name} is missing RGB bands: {missing}. Found: {descriptions}')
        rgb = np.stack([src.read(band_lookup[name]) for name in required], axis=-1)
    return percentile_stretch(rgb)


def build_error_map(truth: np.ndarray, pred: np.ndarray) -> np.ndarray:
    agreement = np.zeros_like(truth, dtype=np.uint8)
    valid = truth != 0
    agreement[valid & (truth == pred)] = 1
    agreement[valid & (truth != pred)] = 2
    return colorize_labels(agreement, ERROR_COLORS)


metrics_df = parse_metrics(run_dir / 'metrics.txt')
metrics_df['label_file'] = [path.name for path in label_files]
metrics_df['prediction_file'] = [path.name for path in pred_files]
metrics_df['before_file'] = [path.name for path in before_files]
metrics_df['after_file'] = [path.name for path in after_files]

In [ ]:
summary_df = metrics_df[['image_id', 'label_file', 'Accuracy', 'Precision', 'Recall', 'F1', 'IoU']].copy()
display(Markdown('## Run Summary'))
display(summary_df.style.format({
    'Accuracy': '{:.4f}',
    'Precision': '{:.4f}',
    'Recall': '{:.4f}',
    'F1': '{:.4f}',
    'IoU': '{:.4f}',
}).background_gradient(subset=['Accuracy', 'Precision', 'Recall', 'F1', 'IoU'], cmap='YlGn'))

overview = pd.DataFrame({
    'Metric': ['Mean Accuracy', 'Mean Precision', 'Mean Recall', 'Mean F1', 'Mean IoU'],
    'Value': [
        metrics_df['Accuracy'].mean(),
        metrics_df['Precision'].mean(),
        metrics_df['Recall'].mean(),
        metrics_df['F1'].mean(),
        metrics_df['IoU'].mean(),
    ],
})
display(Markdown('## Aggregate Holdout Metrics'))
display(overview.style.format({'Value': '{:.4f}'}))

In [ ]:
legend_rows = []
for cls_id, cls_name in CLASS_NAMES.items():
    rgb = CLASS_COLORS[cls_id] / 255.0
    swatch = f"<div style='width:24px;height:24px;background:rgb({CLASS_COLORS[cls_id][0]},{CLASS_COLORS[cls_id][1]},{CLASS_COLORS[cls_id][2]});border:1px solid #999;'></div>"
    legend_rows.append({'Class ID': cls_id, 'Class': cls_name, 'Color': swatch})
legend_df = pd.DataFrame(legend_rows)
display(Markdown('## Label Legend'))
display(legend_df.to_html(escape=False, index=False))
display(Markdown('**Agreement map colors**: white = correct prediction on labeled pixels, blue = disagreement, black = background / ignored area.'))

In [ ]:
tile_idx = 0

before_rgb = read_s2_rgb(before_files[tile_idx])
after_rgb = read_s2_rgb(after_files[tile_idx])
label = read_single_band(label_files[tile_idx]).astype(np.uint8)
prediction = read_single_band(pred_files[tile_idx]).astype(np.uint8)

label_rgb = colorize_labels(label, CLASS_COLORS)
prediction_rgb = colorize_labels(prediction, CLASS_COLORS)
error_rgb = build_error_map(label, prediction)

row = metrics_df.iloc[tile_idx]
display(Markdown(f"## Detailed View: Holdout Tile {tile_idx}"))
display(Markdown(
    f"**Label file:** `{row.label_file}`  \\n+**Before image:** `{row.before_file}`  \\n+**After image:** `{row.after_file}`  \\n+**Prediction file:** `{row.prediction_file}`"
))

fig, axes = plt.subplots(1, 5, figsize=(25, 5))
axes[0].imshow(before_rgb)
axes[0].set_title('Before RGB')
axes[1].imshow(after_rgb)
axes[1].set_title('After RGB')
axes[2].imshow(label_rgb)
axes[2].set_title('Ground Truth')
axes[3].imshow(prediction_rgb)
axes[3].set_title('Prediction')
axes[4].imshow(error_rgb)
axes[4].set_title('Agreement / Error')
for ax in axes:
    ax.axis('off')
fig.suptitle(
    f"Tile {tile_idx} | Acc={row.Accuracy:.4f} | Prec={row.Precision:.4f} | Recall={row.Recall:.4f} | F1={row.F1:.4f} | IoU={row.IoU:.4f}",
    fontsize=14,
    y=1.03,
)
plt.tight_layout()
plt.show()

In [ ]:
display(Markdown('## Full Holdout Gallery'))
n_tiles = len(pred_files)
fig, axes = plt.subplots(n_tiles, 5, figsize=(25, 5 * n_tiles))
if n_tiles == 1:
    axes = np.expand_dims(axes, axis=0)

for tile_idx in range(n_tiles):
    before_rgb = read_s2_rgb(before_files[tile_idx])
    after_rgb = read_s2_rgb(after_files[tile_idx])
    label = read_single_band(label_files[tile_idx]).astype(np.uint8)
    prediction = read_single_band(pred_files[tile_idx]).astype(np.uint8)
    label_rgb = colorize_labels(label, CLASS_COLORS)
    prediction_rgb = colorize_labels(prediction, CLASS_COLORS)
    error_rgb = build_error_map(label, prediction)
    row = metrics_df.iloc[tile_idx]

    panels = [before_rgb, after_rgb, label_rgb, prediction_rgb, error_rgb]
    titles = [
        f'Before RGB\nTile {tile_idx}',
        'After RGB',
        'Ground Truth',
        'Prediction',
        f'Agreement / Error\nIoU={row.IoU:.4f} | F1={row.F1:.4f}',
    ]

    for col_idx, (panel, title) in enumerate(zip(panels, titles)):
        axes[tile_idx, col_idx].imshow(panel)
        axes[tile_idx, col_idx].set_title(title)
        axes[tile_idx, col_idx].axis('off')

plt.tight_layout()
plt.show()